# 03 — Modelagem da Camada Gold

## 1. Objetivo

A camada Gold representa a etapa de organização dos dados para consumo analítico. A partir dos dados tratados e padronizados na camada Silver, será construído um modelo dimensional em esquema estrela (Star Schema), facilitando consultas, análises e a interpretação das informações do mercado imobiliário.

O modelo será composto por uma tabela fato central e tabelas dimensão relacionadas às perspectivas de tempo, localização e tipo de transação.

## 2. Definição do grão

O grão da tabela fato corresponde a uma observação mensal do mercado imobiliário para uma determinada combinação de bairro/localidade e tipo de transação.

Cada registro poderá conter as seguintes métricas:

- mediana do preço por metro quadrado;
- percentil 25 do preço por metro quadrado;
- percentil 75 do preço por metro quadrado;
- quantidade média diária de anúncios ativos;
- quantidade de novos anúncios;
- média de dias dos imóveis no mercado.

## 3. Estrutura do modelo dimensional

O modelo será estruturado em esquema estrela, contendo:

- `dim_tempo`: dimensão temporal para representação dos períodos analisados;
- `dim_localidade`: dimensão contendo cidade, UF e bairro;
- `dim_transacao`: dimensão correspondente ao tipo de transação imobiliária;
- `fato_mercado_imobiliario`: tabela fato contendo as métricas do mercado e as chaves de relacionamento com as dimensões.

Essa separação permite organizar as informações de forma adequada para análises multidimensionais e reduz a repetição dos atributos descritivos na tabela fato.

## 3.1 Leitura da camada Silver

A construção do modelo dimensional utiliza como fonte a tabela tratada da camada Silver. Nesta etapa, os dados previamente limpos e padronizados são carregados para a criação das dimensões e da tabela fato.

Antes da modelagem, será validada a quantidade de registros recebidos da camada Silver, garantindo a rastreabilidade entre as etapas do pipeline.

In [0]:
# Leitura da tabela tratada da camada Silver
tabela_silver = "mvp_imobiliario.silver.mercado_imobiliario_tratado"

df_silver = spark.table(tabela_silver)

# Validação da quantidade de registros
total_silver = df_silver.count()

print(f"Registros recebidos da camada Silver: {total_silver}")

# Visualização de uma amostra dos dados
display(df_silver.limit(10))

Registros recebidos da camada Silver: 36933


mes,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio,arquivo_origem,cidade,uf,data_ingestao
2025-09-01,Abranches,venda,5941,4341,6781,42,1,511,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Água Verde,aluguel,50,41,59,4,0,286,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Água Verde,venda,11771,9692,14377,533,11,384,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Ahú,aluguel,48,48,48,2,0,587,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Ahú,venda,12235,11027,13344,126,1,352,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto Boqueirão,aluguel,25,24,25,3,0,144,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto Boqueirão,venda,4866,4375,5513,58,1,366,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Glória,aluguel,72,52,86,14,0,784,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Glória,venda,12762,10920,15346,86,1,443,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Rua XV,aluguel,26,26,26,1,0,104,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z


### 3.2 Dimensão Tempo — `dim_tempo`

A dimensão tempo representa os períodos mensais presentes no conjunto de dados e permite a realização de análises temporais das métricas do mercado imobiliário.

A partir do atributo `mes` da camada Silver, serão obtidos os períodos distintos existentes na base. Para cada período será criada uma chave substituta (`id_tempo`), utilizada posteriormente no relacionamento com a tabela fato.

Além da data de referência, serão disponibilizados os atributos de ano e número do mês, facilitando agrupamentos e análises temporais.

In [0]:
from pyspark.sql.functions import col, year, month, date_format, row_number
from pyspark.sql.window import Window

# Criação da dimensão Tempo
dim_tempo = (
    df_silver
    .select("mes")
    .distinct()
    .withColumn("ano", year(col("mes")))
    .withColumn("numero_mes", month(col("mes")))
    .withColumn("ano_mes", date_format(col("mes"), "yyyy-MM"))
)

# Criação da chave substituta
janela_tempo = Window.orderBy("mes")

dim_tempo = (
    dim_tempo
    .withColumn("id_tempo", row_number().over(janela_tempo))
    .select(
        "id_tempo",
        "mes",
        "ano",
        "numero_mes",
        "ano_mes"
    )
    .orderBy("mes")
)

print(f"Quantidade de períodos na dimensão Tempo: {dim_tempo.count()}")

display(dim_tempo)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Quantidade de períodos na dimensão Tempo: 13


id_tempo,mes,ano,numero_mes,ano_mes
1,2025-09-01,2025,9,2025-09
2,2025-10-01,2025,10,2025-10
3,2025-11-01,2025,11,2025-11
4,2025-12-01,2025,12,2025-12
5,2026-01-01,2026,1,2026-01
6,2026-02-01,2026,2,2026-02
7,2026-03-01,2026,3,2026-03
8,2026-04-01,2026,4,2026-04
9,2026-05-01,2026,5,2026-05
10,2026-06-01,2026,6,2026-06


### 3.3 Dimensão Localidade — `dim_localidade`

A dimensão localidade organiza os atributos geográficos presentes no conjunto de dados, permitindo analisar as métricas do mercado imobiliário sob diferentes perspectivas de localização.

Cada combinação distinta de cidade, Unidade Federativa (UF) e bairro representa uma localidade do modelo. Para cada localidade será criada uma chave substituta (`id_localidade`), utilizada posteriormente no relacionamento com a tabela fato.

A dimensão será composta pelos seguintes atributos:

- `id_localidade`: chave substituta da dimensão;
- `cidade`: cidade à qual o bairro pertence;
- `uf`: Unidade Federativa correspondente;
- `bairro`: bairro ao qual as métricas imobiliárias se referem.

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Criação da dimensão Localidade
dim_localidade = (
    df_silver
    .select(
        "cidade",
        "uf",
        "bairro"
    )
    .distinct()
)

# Criação da chave substituta
janela_localidade = Window.orderBy("uf", "cidade", "bairro")

dim_localidade = (
    dim_localidade
    .withColumn(
        "id_localidade",
        row_number().over(janela_localidade)
    )
    .select(
        "id_localidade",
        "cidade",
        "uf",
        "bairro"
    )
    .orderBy("id_localidade")
)

print(
    f"Quantidade de localidades na dimensão Localidade: "
    f"{dim_localidade.count()}"
)

display(dim_localidade)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Quantidade de localidades na dimensão Localidade: 2129


id_localidade,cidade,uf,bairro
1,salvador,ba,ALPHAVILLE2
2,salvador,ba,Acupe
3,salvador,ba,Acupe de Brotas
4,salvador,ba,Alphaville 2
5,salvador,ba,Alphaville I
6,salvador,ba,Alphaville II
7,salvador,ba,Alto das Pombas
8,salvador,ba,Alto do Cabrito
9,salvador,ba,Alto do Coqueirinho
10,salvador,ba,Alto do Peru


### 3.4 Dimensão Transação — `dim_transacao`

A dimensão Transação representa os tipos de negociação imobiliária existentes no conjunto de dados.

A análise realizada na camada Silver identificou dois tipos de transação: **venda** e **aluguel**. Para cada tipo será criada uma chave substituta (`id_transacao`), que será utilizada posteriormente no relacionamento com a tabela fato.

A dimensão será composta pelos seguintes atributos:

- `id_transacao`: chave substituta da dimensão;
- `transacao`: tipo de transação imobiliária.

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Criação da dimensão Transação
dim_transacao = (
    df_silver
    .select("transacao")
    .distinct()
)

# Criação da chave substituta
janela_transacao = Window.orderBy("transacao")

dim_transacao = (
    dim_transacao
    .withColumn(
        "id_transacao",
        row_number().over(janela_transacao)
    )
    .select(
        "id_transacao",
        "transacao"
    )
    .orderBy("id_transacao")
)

print(
    f"Quantidade de tipos de transação na dimensão Transação: "
    f"{dim_transacao.count()}"
)

display(dim_transacao)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Quantidade de tipos de transação na dimensão Transação: 2


id_transacao,transacao
1,aluguel
2,venda


### 3.5 Construção da tabela fato — `fato_mercado_imobiliario`

A tabela fato representa as observações mensais do mercado imobiliário e concentra as métricas quantitativas utilizadas nas análises do projeto.

Conforme o grão definido para o modelo, cada registro da tabela fato representa uma combinação de **período, localidade e tipo de transação**. Os atributos descritivos da camada Silver serão substituídos pelas respectivas chaves das dimensões `dim_tempo`, `dim_localidade` e `dim_transacao`.

A tabela fato será composta pelas seguintes chaves:

- `id_tempo`: referência à dimensão temporal;
- `id_localidade`: referência à dimensão de localidade;
- `id_transacao`: referência à dimensão de tipo de transação.

E pelas seguintes métricas:

- `mediana_m2`: valor mediano do preço por metro quadrado;
- `p25_m2`: percentil 25 do preço por metro quadrado;
- `p75_m2`: percentil 75 do preço por metro quadrado;
- `anuncios_ativos_media_dia`: quantidade média diária de anúncios ativos;
- `novos_anuncios`: quantidade de novos anúncios no período;
- `dias_no_mercado_medio`: quantidade média de dias dos imóveis no mercado.

A construção será realizada por meio do relacionamento da tabela Silver com as três dimensões previamente criadas.

In [0]:
# Construção da tabela fato a partir da camada Silver
# Relacionamento com as dimensões Tempo, Localidade e Transação

fato_mercado_imobiliario = (
    df_silver.alias("s")

    # Relacionamento com a dimensão Tempo
    .join(
        dim_tempo.alias("dt"),
        col("s.mes") == col("dt.mes"),
        "left"
    )

    # Relacionamento com a dimensão Localidade
    .join(
        dim_localidade.alias("dl"),
        (col("s.cidade") == col("dl.cidade")) &
        (col("s.uf") == col("dl.uf")) &
        (col("s.bairro") == col("dl.bairro")),
        "left"
    )

    # Relacionamento com a dimensão Transação
    .join(
        dim_transacao.alias("dtr"),
        col("s.transacao") == col("dtr.transacao"),
        "left"
    )

    # Seleção das chaves dimensionais e métricas
    .select(
        col("dt.id_tempo"),
        col("dl.id_localidade"),
        col("dtr.id_transacao"),
        col("s.mediana_m2"),
        col("s.p25_m2"),
        col("s.p75_m2"),
        col("s.anuncios_ativos_media_dia"),
        col("s.novos_anuncios"),
        col("s.dias_no_mercado_medio")
    )
)

print(f"Registros na camada Silver: {df_silver.count()}")
print(f"Registros na tabela fato: {fato_mercado_imobiliario.count()}")

display(fato_mercado_imobiliario.limit(10))

Registros na camada Silver: 36933


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Registros na tabela fato: 36933


id_tempo,id_localidade,id_transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio
1,268,2,5941,4341,6781,42,1,511
1,348,1,50,41,59,4,0,286
1,348,2,11771,9692,14377,533,11,384
1,269,1,48,48,48,2,0,587
1,269,2,12235,11027,13344,126,1,352
1,270,1,25,24,25,3,0,144
1,270,2,4866,4375,5513,58,1,366
1,271,1,72,52,86,14,0,784
1,271,2,12762,10920,15346,86,1,443
1,272,1,26,26,26,1,0,104


#### 3.5.1 Validação da integridade dos relacionamentos dimensionais

Após a construção da tabela fato, será verificada a integridade dos relacionamentos com as dimensões.

Além da manutenção da quantidade total de registros entre a camada Silver e a tabela fato, é necessário confirmar que todos os registros encontraram correspondência nas dimensões Tempo, Localidade e Transação.

Para isso, será verificada a existência de valores nulos nas chaves dimensionais `id_tempo`, `id_localidade` e `id_transacao`.


In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Verificação de chaves dimensionais sem correspondência
validacao_chaves = fato_mercado_imobiliario.select(
    spark_sum(
        when(col("id_tempo").isNull(), 1).otherwise(0)
    ).alias("id_tempo_nulos"),

    spark_sum(
        when(col("id_localidade").isNull(), 1).otherwise(0)
    ).alias("id_localidade_nulos"),

    spark_sum(
        when(col("id_transacao").isNull(), 1).otherwise(0)
    ).alias("id_transacao_nulos")
)

display(validacao_chaves)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_tempo_nulos,id_localidade_nulos,id_transacao_nulos
0,0,0


### 3.6 Persistência do modelo dimensional na camada Gold

Após a construção e validação do modelo dimensional, as três dimensões e a tabela fato serão persistidas no schema `gold` do Unity Catalog em formato Delta.

O modelo é composto pelas seguintes tabelas:

- `dim_tempo`;
- `dim_localidade`;
- `dim_transacao`;
- `fato_mercado_imobiliario`.

A persistência em tabelas gerenciadas permite disponibilizar a camada Gold para consultas analíticas, mantendo a separação entre os dados brutos da Bronze, os dados tratados da Silver e os dados modelados para consumo na Gold.

In [0]:
# Persistência do modelo dimensional na camada Gold

tabelas_gold = {
    "dim_tempo": dim_tempo,
    "dim_localidade": dim_localidade,
    "dim_transacao": dim_transacao,
    "fato_mercado_imobiliario": fato_mercado_imobiliario
}

for nome_tabela, dataframe in tabelas_gold.items():

    tabela_destino = f"mvp_imobiliario.gold.{nome_tabela}"

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )

    print(f"Tabela criada com sucesso: {tabela_destino}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela criada com sucesso: mvp_imobiliario.gold.dim_tempo
Tabela criada com sucesso: mvp_imobiliario.gold.dim_localidade
Tabela criada com sucesso: mvp_imobiliario.gold.dim_transacao
Tabela criada com sucesso: mvp_imobiliario.gold.fato_mercado_imobiliario


### 3.7 Validação das tabelas persistidas na camada Gold

Após a persistência do modelo dimensional, as tabelas armazenadas no Unity Catalog são novamente consultadas para validar sua criação e a quantidade de registros.

Essa verificação permite confirmar que o modelo dimensional foi efetivamente materializado na camada Gold e está disponível para utilização nas etapas de análise.

In [0]:
# Validação das tabelas persistidas na camada Gold

tabelas_validacao = [
    "dim_tempo",
    "dim_localidade",
    "dim_transacao",
    "fato_mercado_imobiliario"
]

for tabela in tabelas_validacao:
    nome_completo = f"mvp_imobiliario.gold.{tabela}"
    quantidade = spark.table(nome_completo).count()

    print(f"{tabela}: {quantidade} registros")

dim_tempo: 13 registros
dim_localidade: 2129 registros
dim_transacao: 2 registros
fato_mercado_imobiliario: 36933 registros


In [0]:
# ============================================================
# DOCUMENTAÇÃO DO CATÁLOGO DE DADOS - CAMADA GOLD
# ============================================================

# ------------------------------------------------------------
# 1. DIMENSÃO TEMPO
# ------------------------------------------------------------

spark.sql("""
COMMENT ON TABLE mvp_imobiliario.gold.dim_tempo IS
'Dimensão temporal do modelo dimensional do mercado imobiliário. Cada registro representa um mês presente no período analisado.'
""")

comentarios_dim_tempo = {
    "id_tempo": "Chave substituta (surrogate key) da dimensão Tempo.",
    "mes": "Mês de referência das métricas do mercado imobiliário.",
    "ano": "Ano correspondente ao período analisado.",
    "numero_mes": "Número do mês, variando de 1 a 12.",
    "ano_mes": "Identificação do período no formato AAAA-MM."
}

for coluna, descricao in comentarios_dim_tempo.items():
    spark.sql(f"""
        ALTER TABLE mvp_imobiliario.gold.dim_tempo
        ALTER COLUMN `{coluna}`
        COMMENT '{descricao}'
    """)


# ------------------------------------------------------------
# 2. DIMENSÃO LOCALIDADE
# ------------------------------------------------------------

spark.sql("""
COMMENT ON TABLE mvp_imobiliario.gold.dim_localidade IS
'Dimensão geográfica do modelo dimensional. Contém as localidades presentes nos dados do mercado imobiliário, identificadas por cidade, unidade federativa e bairro.'
""")

comentarios_dim_localidade = {
    "id_localidade": "Chave substituta (surrogate key) da dimensão Localidade.",
    "cidade": "Cidade à qual pertence a localidade analisada.",
    "uf": "Unidade Federativa correspondente à cidade.",
    "bairro": "Bairro ao qual as métricas do mercado imobiliário se referem."
}

for coluna, descricao in comentarios_dim_localidade.items():
    spark.sql(f"""
        ALTER TABLE mvp_imobiliario.gold.dim_localidade
        ALTER COLUMN `{coluna}`
        COMMENT '{descricao}'
    """)


# ------------------------------------------------------------
# 3. DIMENSÃO TRANSAÇÃO
# ------------------------------------------------------------

spark.sql("""
COMMENT ON TABLE mvp_imobiliario.gold.dim_transacao IS
'Dimensão que representa os tipos de transação imobiliária presentes no conjunto de dados.'
""")

comentarios_dim_transacao = {
    "id_transacao": "Chave substituta (surrogate key) da dimensão Transação.",
    "transacao": "Tipo de transação imobiliária, podendo representar venda ou aluguel."
}

for coluna, descricao in comentarios_dim_transacao.items():
    spark.sql(f"""
        ALTER TABLE mvp_imobiliario.gold.dim_transacao
        ALTER COLUMN `{coluna}`
        COMMENT '{descricao}'
    """)


# ------------------------------------------------------------
# 4. TABELA FATO
# ------------------------------------------------------------

spark.sql("""
COMMENT ON TABLE mvp_imobiliario.gold.fato_mercado_imobiliario IS
'Tabela fato do modelo dimensional. Cada registro representa uma observação mensal do mercado imobiliário para uma combinação de localidade e tipo de transação, contendo as métricas utilizadas nas análises.'
""")

comentarios_fato = {
    "id_tempo": "Chave de relacionamento com a dimensão Tempo.",
    "id_localidade": "Chave de relacionamento com a dimensão Localidade.",
    "id_transacao": "Chave de relacionamento com a dimensão Transação.",
    "mediana_m2": "Valor mediano do preço por metro quadrado.",
    "p25_m2": "Percentil 25 do preço por metro quadrado.",
    "p75_m2": "Percentil 75 do preço por metro quadrado.",
    "anuncios_ativos_media_dia": "Quantidade média diária de anúncios ativos.",
    "novos_anuncios": "Quantidade de novos anúncios registrados no período.",
    "dias_no_mercado_medio": "Quantidade média de dias dos imóveis no mercado."
}

for coluna, descricao in comentarios_fato.items():
    spark.sql(f"""
        ALTER TABLE mvp_imobiliario.gold.fato_mercado_imobiliario
        ALTER COLUMN `{coluna}`
        COMMENT '{descricao}'
    """)

print("Documentação das tabelas e colunas da camada Gold concluída com sucesso.")

Documentação das tabelas e colunas da camada Gold concluída com sucesso.
